H = (p_r · v) + (p_v · a_grav) + (p_v · a_solar) + 2 (a_solar · v)

dp_r/dt = -∂H/∂r
dp_v/dt = -∂H/∂v

In [ ]:
# Константы
AU = 149597870.7  # км
R_sun = 700000  # км
MU_sun = 132712440019.  # км ^ 3 / с ^2

R_marks = 227e6
# Единицы измерения
DU = AU  # Задано в км
TU = 60 * 60 * 24.  # Задано в секундах; сейчас это 1 день
VU = DU / TU
CU = DU / (TU ** 2)  # км / с ^ 2

# Константы
MU_in_units = MU_sun * (TU ** 2 / DU ** 3)  # DU^3/TU^2
AU_in_units = AU / DU
R_sun_in_units = R_sun / DU
Marks_orbit_radius_in_units = R_marks / DU

c_light = 299_792_458  # m/s
E_Earth = 1372.  # Wt/m^2 = kg / s^3
P_Earth = E_Earth / c_light  # kg / (m * s^2)

# solar pressure at one astronomical unit (AU)
P_au = P_Earth * TU ** 2 / (DU * 1e3)  # N_in_units / m^2 = (kg * DU / TU^2) / m^2


# Начальные параметры 
import numpy as np

# коэффициент отражения
r = 0.91

# non-Lambertian coefficients
B_f = 0.79  # front
B_b = 0.67  # back
e_f = 0.025  # излучение от передней поверхности
e_b = 0.27
s = 1

# Время интегрирования
t_span = (0, 250 * 24 * 60 * 60/ TU)
t_eval = np.linspace(t_span[0], t_span[1], num= 100_000)


In [ ]:
# imports 
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
import plotly.graph_objects as go
import h5py

In [ ]:
# Функция конвертации 
def orbital_elements_to_state(a, e, i, Omega, omega, nu, mu):
    # a - большая полуось (м)
    # e - эксцентриситет
    # i - наклонение (рад)
    # Omega - долгота восходящего узла (рад)
    # omega - аргумент перицентра (рад)
    # nu - истинная аномалия (рад)
    # mu - гравитационный параметр (м^3/с^2)

    # 1. Вычисление радиус-вектора r

    p = a * (1 - e * e)
    distance = p / (1 + e * np.cos(nu))

    # 2. Вектор в орбитальной системе координат
    r_orbital = np.array([1, 0, 0]) * distance

    # 3. Вектор скорости в орбитальной системе координат

    velocity = np.array([np.sqrt(mu / p) * e * np.sin(nu),
                         np.sqrt(mu / p) * (1 + e * np.cos(nu)),
                         0])

    # 4. Преобразование вектора в инерциальную систему координат
    R3_W = np.array([[np.cos(Omega), np.sin(Omega), 0],
                     [-np.sin(Omega), np.cos(Omega), 0],
                     [0, 0, 1]])
    R1_i = np.array([[1, 0, 0], [0, np.cos(i), np.sin(i)], [0, np.sin(i), np.cos(i)]])

    u = omega + nu
    R3_w = np.array([[np.cos(u), np.sin(u), 0],
                     [-np.sin(u), np.cos(u), 0],
                     [0, 0, 1]])
    R = R3_W @ R1_i @ R3_w

    # 5. Преобразование радиус-вектора и вектора скорости
    r_inertial = R.T @ r_orbital
    v_inertial = R.T @ velocity

    return r_inertial, v_inertial

In [ ]:
# GravityForce

import numpy as np
def gravity_acceleration(r: np.array) -> np.array:
    # r должно быть в DU
    global MU_in_units
    r_magnitude = np.linalg.norm(r)
    return - MU_in_units * r / (r_magnitude ** 3)

In [ ]:
# Вычисление угла управления и SolarForce
def angles_for_max_projection(p: np.ndarray) -> tuple[float, float]:
    """
    Вычисляет оптимальные углы альфа и дельта паруса для максимизации проекции силы тяги
    на произвольное направление, заданное вектором p.
    
    :param p: Направляющий вектор в ОСК
    :return: Оптимальные углы в радианах
    """
    if p.shape != (3,):
        raise ValueError("Вектор p должен быть размерности (3,)")

    # Нормализуем вектор p
    p = p / np.linalg.norm(p)

    # Проверяем диапазон для arccos
    cos_alpha =  p[0]
    cos_alpha = np.clip(cos_alpha, -1.0, 1.0)
    
    p_alpha = p - np.array([cos_alpha, 0, 0])
    cos_delta = p_alpha @ np.array([0, 1, 0])
    sin_delta = p_alpha @ np.array([0, 0, 1])
    
    delta = np.atan2(sin_delta, cos_delta)
    # print("Alpha:", np.arccos(cos_alpha) * 180 / np.pi)
    # print("Delta:", delta * 180 / np.pi)
    # Оптимальные углы для нормали
    cos_psi = cos_alpha # TO DO: remove psi; rename angle so as to distinguish angles corresponding do different vectors
    sqrt_cos_psi = cos_psi * np.sqrt(8 + cos_psi ** 2)

    cos2_a = (8 / 3) * (3 + sqrt_cos_psi) / (12 + cos_psi ** 2 + sqrt_cos_psi)
    cos2_a = np.clip(cos2_a, 0.0, 1.0)  # cos²(a) должен быть в [0,1]

    cos_a = np.sqrt(cos2_a)
    alpha_opt = np.arccos(cos_a)
    delta_opt = delta  # Оптимально совпадает с направлением

    return alpha_opt, delta_opt

# Solar force 
import math

def solar_force(r_i: np.array, v_i: np.array, p_v: np.array, A_del_m_: float) -> np.array:

    """
    Вычисляет силу давления солнечного света на солнечный парус.

    Параметры:
        r_i (np.array): радиус-вектор спутника в ИСК (м).
        v_i (np.array): вектор скорости спутника в ИСК (м/с).
        g   (np.array): направление, на которое максимизируем проекцию ускорения
        psi (float): угол между скоростью и радиусом-вектором (рад).
        delta (float): дополнительный угол управления (рад).

    Возвращает:
        np.array: сила давления солнечного света в ИСК (Н/кг).
    """

    # Вектор углового момента и его модуль
    c_i = np.cross(r_i, v_i)
    c_i_norm = np.linalg.norm(c_i)

    # Нормализация радиус-вектора
    r_norm = np.linalg.norm(r_i)
    e1 = r_i / r_norm
    e3 = c_i / c_i_norm
    e2 = np.cross(e3, e1)

    # Матрица перехода из ОСК в ИСК
    transformation_matrix = np.array([e1, e2, e3]).T

    # Скорость в ОСК
    v_orbital = transformation_matrix.T @ v_i

    g = p_v + 2 * v_i

    # Углы управления α, delta
    alpha, delta = angles_for_max_projection(-g)

    # Нормаль к поверхности паруса в орбитальной СК
    sail_normal = np.array([
        np.cos(alpha),
        np.cos(delta) * np.sin(alpha),
        np.sin(delta) * np.sin(alpha)
    ]) 
   
    sail_normal_e = sail_normal / np.linalg.norm(sail_normal)
    # Направление солнечного света в орбитальной СК
    solar_direction = np.array([1, 0, 0])
    cos_a = np.dot(sail_normal_e, solar_direction)
    sin_a = np.sqrt(1 - cos_a ** 2)

    # Давление солнечного света
    pressure = P_au * (AU_in_units / r_norm) ** 2

    # Компоненты силы давления
    f_n = pressure * (
        (1 + r * s) * cos_a ** 2 +
        B_f * (1 - s) * r * cos_a +
        (1 - r) * ((e_f * B_f - e_b * B_b) / (e_f + e_b)) * cos_a
    )
    f_t = pressure * (1 - r * s) * cos_a * sin_a

    #sail_tranversal
    # Сила в орбитальной СК
    force_orbital = f_n * sail_normal + f_t * np.array([0, 1, 0])

    # Сила в инерциальной СК
    force_inertial = np.dot(transformation_matrix, force_orbital)

    return force_inertial * A_del_m_

In [ ]:
# hamiltonian_derivatives

a_c = np.array([2 / (CU * 1e6)])

m_del_A = P_au * ((1 + r * s) + B_f * (1 - s) * r + (1 - r) *
                                    ((e_f * B_f - e_b * B_b) / (e_f + e_b))) / a_c

A_del_m = 1 / m_del_A

def hamiltonian_derivatives(t, y, mu):
    """Derivatives for the Hamiltonian system"""
    r = y[:3]
    v = y[3:6]
    p_r = y[6:9]
    p_v = y[9:12]
    
    r_norm = np.linalg.norm(r)

    a_grav = gravity_acceleration(r)
   
    a_solar = solar_force(r, v, p_v, A_del_m)
    drdt = v
    dvdt = a_grav + a_solar
    
    dp_rdt = p_v @ (mu * (np.eye(3)/r_norm**3 - 3 * np.outer(r, r)/r_norm**5))
    dp_vdt = -p_r - 2*a_solar
    
    return np.concatenate([drdt, dvdt, dp_rdt, dp_vdt])

In [ ]:
# event of stop
def mars_orbit_reached(t, y):
    r = y[:3]
    return np.linalg.norm(r) - Marks_orbit_radius_in_units
mars_orbit_reached.terminal = True
mars_orbit_reached.direction = 1

In [ ]:
# основная функция симуляции

def simulate_trajectory(p0, initial_state, t_span, mu):
    y0 = np.concatenate([initial_state, p0])
    
    sol = solve_ivp(
        lambda t, y: hamiltonian_derivatives(t, y, mu),
        t_span,
        y0,
        events=mars_orbit_reached,
        dense_output=True,
        rtol=1e-8,
        atol=1e-8
    )
    return sol

def objective(p0, initial_state, t_span, mu, target_radius):
    """Objective function for optimization"""
    sol = simulate_trajectory(p0, initial_state, t_span, mu)
    
    if sol.t_events[0].size > 0:
        return 0.0
    else:
        final_r = sol.y[:3, -1]
        return np.abs(np.linalg.norm(final_r) - target_radius) * 1e6

In [ ]:
import h5py
import numpy as np
import pandas as pd

def calculate_orbital_elements(v, r_p_f):
    """Вычисляет большую полуось и эксцентриситет орбиты."""
    a = -MU_in_units / (v ** 2)
    e = 1 - r_p_f / a
    
    return a, e

def hdf5_to_csv(hdf5_file):
    # Открываем HDF5 файл
    with h5py.File(hdf5_file, 'r') as f:
        # Создаем список для хранения данных
        data_list = []
        
        # Получаем группу с траекториями
        trajectories_group = f['trajectories']

        trajectory_keys = list(trajectories_group.keys())[:10]
        # Проходим по всем траекториям
        for traj_name in trajectory_keys:
            traj_group = trajectories_group[traj_name]
            
            # Извлекаем данные
            x = traj_group['x'][:]
            y = traj_group['y'][:]
            z = traj_group['z'][:]
            vx = traj_group['vx'][:]
            vy = traj_group['vy'][:]
            vz = traj_group['vz'][:]
            time = traj_group['time'][:]
            a_c = traj_group['a_c'][()]
            v_inf = traj_group['v_inf'][()]
            r_p = traj_group['r_p'][()]
            r_min = traj_group['r_min'][()]
            
            # Начальная и конечная позиции
            x0 = np.array([x[0], y[0], z[0]])
            xf = np.array([x[-1], y[-1], z[-1]])
            r_p_f = r_p
            
            # Конечная скорость
            vf = np.array([vx[-1], vy[-1], vz[-1]])
            
            # Вычисляем конечные орбитальные элементы
            # Замените mu на ваше значение гравитационного параметра
          
            a_f, e_f = calculate_orbital_elements(vf, r_p_f)
            
            # Добавляем данные в список
            data_list.append({
                'trajectory': traj_name,
                'x0_x': x0[0],
                'x0_y': x0[1],
                'x0_z': x0[2],
                'xf_x': xf[0],
                'xf_y': xf[1],
                'xf_z': xf[2],
                'vxf': vf[0],
                'vyf': vf[1],
                'vzf': vf[2],
                'time': time[-1],
                'a_c': a_c,
                'v_inf': v_inf,
                'r_p': r_p,
                'r_min': r_min,
                'a_f': a_f,
                'e_f': e_f
            })
    return data_list

# Пример использования
data = hdf5_to_csv('trajectories_big_data.h5')

In [ ]:
# начальные параметры 

first = data[1]

initial_r = np.array([first['xf_x'], first['xf_y'], first['xf_z']])
initial_v = np.array([-first['vxf'], -first['vyf'], -first['vzf']])
initial_state = np.concatenate([initial_r, initial_v])
t_span = (0, 365 * 24 * 3600 * 2)

p0_guess = np.array([0.0, 0.0, 0.0, 0.0, -1.0, 0.0])

print("Optimizing trajectory...")
result = minimize(
    objective,
    p0_guess,
    args=(initial_state, t_span, MU_in_units, Marks_orbit_radius_in_units),
    method='Nelder-Mead',
    options={'maxiter': 100, 'disp': True}
)

p0_optimal = result.x

In [ ]:
optimal_sol = simulate_trajectory(p0_optimal, initial_state, t_span, MU_in_units)

In [ ]:
# функции для построения графиков 
fig = go.Figure()

# Trajectory
fig.add_trace(go.Scatter(
    x=optimal_sol.y[0],
    y=optimal_sol.y[1],
    mode='lines',
    name='Spacecraft Trajectory',
    line=dict(color='blue', width=2)
))

# Sun
fig.add_trace(go.Scatter(
    x=[0],
    y=[0],
    mode='markers',
    name='Sun',
    marker=dict(color='yellow', size=10)
))

# Earth orbit (circular)
theta = np.linspace(0, 2*np.pi, 100)
earth_orbit_x = np.cos(theta)
earth_orbit_y = np.sin(theta)
fig.add_trace(go.Scatter(
    x=earth_orbit_x,
    y=earth_orbit_y,
    mode='lines',
    name='Earth Orbit',
    line=dict(color='green', dash='dash')
))

# Mars orbit
mars_orbit_x = 1.52 * np.cos(theta)
mars_orbit_y = 1.52 * np.sin(theta)
fig.add_trace(go.Scatter(
    x=mars_orbit_x,
    y=mars_orbit_y,
    mode='lines',
    name='Mars Orbit',
    line=dict(color='red', dash='dash')
))

# Final point

final_x = optimal_sol.y[0, -1]
final_y = optimal_sol.y[1, -1]
fig.add_trace(go.Scatter(
    x=[final_x],
    y=[final_y],
    mode='markers',
    name='Arrival at Mars',
    marker=dict(color='red', size=8)
))

first_x = optimal_sol.y[0, 0]
first_y = optimal_sol.y[1, 0]
fig.add_trace(go.Scatter(
    x=[first_x],
    y=[first_y],
    mode='markers',
    name='Starting point',
    marker=dict(color='green', size=8)
))

# Layout configuration
fig.update_layout(
    title='Optimal Trajectory from Earth to Mars (XY Plane)',
    xaxis_title='X Position (AU)',
    yaxis_title='Y Position (AU)',
    showlegend=True,
    width=800,
    height=800,
    template='plotly_dark'
)

# Equal aspect ratio
fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

fig.show()